# Capstone: The Refactoring Arc

This is the capstone of Day 2. We take the warm-up code from notebook 00 and progressively improve it, applying everything from the course.

Each step builds on the last. There are exercise versions (you do the work) and filled versions to check against.

## The Journey

| Step | What changes |
|---|---|
| v1 | Original spaghetti notebook |
| v2 | Wrap everything in a class |
| v3 | Add error handling and validation |
| v4 | Extract service classes |
| v5 | Add configuration with dataclasses |
| v6 | Add docstrings, type hints, logging |
| v7 | *Your call: what would you do next?* |

---
## Step 1: The Original Code

This is the warm-up code from notebook 00. Read through it. You've seen it before. Now you have the vocabulary to fix it.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# --- Original: everything loose ---

np.random.seed(42)
df = pd.DataFrame({
    "customerID": [f"C{i:04d}" for i in range(200)],
    "tenure":     np.random.randint(1, 72, 200).astype(float),
    "MonthlyCharges": np.random.uniform(20, 100, 200),
    "TotalCharges":   np.random.uniform(100, 5000, 200).astype(str),
    "gender":     np.random.choice(["Male", "Female"], 200),
    "Partner":    np.random.choice(["Yes", "No"], 200),
    "Dependents": np.random.choice(["Yes", "No"], 200),
    "PhoneService": np.random.choice(["Yes", "No"], 200),
    "Churn":      np.random.choice(["Yes", "No"], 200),
})
# Inject a few missing values
df.loc[df.sample(10, random_state=1).index, "tenure"] = np.nan

print("Loaded:", df.shape)

In [ ]:
# Validate columns
if not all(col in df.columns for col in ["customerID", "tenure", "Churn"]):
    print("Missing some columns...")

# Encode
le = LabelEncoder()
df2 = df.drop(columns=["customerID"])
for col in ["gender", "Partner", "Dependents", "PhoneService", "Churn"]:
    df2[col] = le.fit_transform(df2[col].astype(str))

# Fix TotalCharges
df2["TotalCharges"] = pd.to_numeric(df2["TotalCharges"], errors="coerce")
df2 = df2.dropna()
print("After cleaning:", df2.shape)

In [ ]:
# Train
features = ["gender", "tenure", "MonthlyCharges", "TotalCharges"]
X = df2[features]
y = df2["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

---
## Step 2: Wrap in a Class

**Goal**: bundle related state into a `ChurnPipeline` class. One object, not 12 loose variables.

**Exercise**: Fill in the `TODO` sections below.

In [ ]:
class ChurnPipeline:
    """TODO: add a class docstring."""

    CATEGORICAL_COLS = ["gender", "Partner", "Dependents", "PhoneService", "Churn"]
    FEATURE_COLS = ["gender", "tenure", "MonthlyCharges", "TotalCharges"]
    TARGET_COL = "Churn"

    def __init__(self):
        self.model = None
        self._encoder = LabelEncoder()
        self._df = None  # cleaned dataframe

    def fit(self, df: pd.DataFrame) -> "ChurnPipeline":
        """TODO: implement — clean, encode, train. Return self."""
        pass

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """TODO: implement — return predictions."""
        pass

    def report(self, X: pd.DataFrame, y: pd.Series) -> None:
        """TODO: implement — print classification_report."""
        pass

    def _clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """TODO: implement — drop ID col, fix TotalCharges, drop NaN."""
        pass

    def _encode(self, df: pd.DataFrame) -> pd.DataFrame:
        """TODO: implement — label encode categorical columns."""
        pass


# Test
pipeline = ChurnPipeline()
pipeline.fit(df)
print("Pipeline fitted successfully")

> <details>
> <summary>Step 2 Solution</summary>
>
> ```python
> def fit(self, df: pd.DataFrame) -> "ChurnPipeline":
>     df = self._clean(df)
>     df = self._encode(df)
>     self._df = df
>     X = df[self.FEATURE_COLS]
>     y = df[self.TARGET_COL]
>     X_tr, _, y_tr, _ = train_test_split(X, y, test_size=0.2, random_state=42)
>     self.model = RandomForestClassifier(n_estimators=100, random_state=42)
>     self.model.fit(X_tr, y_tr)
>     return self
>
> def predict(self, X):
>     if self.model is None:
>         raise RuntimeError("Call fit() first.")
>     return self.model.predict(X)
>
> def report(self, X, y):
>     print(classification_report(y, self.predict(X)))
>
> def _clean(self, df):
>     df = df.drop(columns=["customerID"], errors="ignore").copy()
>     df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
>     return df.dropna()
>
> def _encode(self, df):
>     df = df.copy()
>     for col in self.CATEGORICAL_COLS:
>         if col in df.columns:
>             df[col] = LabelEncoder().fit_transform(df[col].astype(str))
>     return df
> ```
> </details>

---
## Step 3: Add Validation and Error Handling

**Goal**: Make failures loud and informative instead of silent or cryptic.

**Exercise**: Add error handling so the pipeline raises descriptive errors for bad input.

In [ ]:
# Define the exception hierarchy first
class PipelineError(Exception):
    """Base for all pipeline errors."""

class SchemaValidationError(PipelineError):
    def __init__(self, missing_cols: list):
        super().__init__(f"Missing required columns: {missing_cols}")
        self.missing_cols = missing_cols

class DataQualityError(PipelineError):
    pass


# Now update the pipeline to use them
class ChurnPipelineV3(ChurnPipeline):
    REQUIRED_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
    MAX_MISSING_PCT = 0.15

    def fit(self, df: pd.DataFrame) -> "ChurnPipelineV3":
        """TODO: call _validate_schema and _check_missing before cleaning."""
        return super().fit(df)

    def _validate_schema(self, df: pd.DataFrame) -> None:
        """TODO: raise SchemaValidationError if REQUIRED_COLS are missing."""
        pass

    def _check_missing(self, df: pd.DataFrame) -> None:
        """TODO: raise DataQualityError if any required column exceeds MAX_MISSING_PCT."""
        pass


# Test happy path
pv3 = ChurnPipelineV3()
pv3.fit(df)
print("Happy path: OK")

# Test schema error
bad_df = df.drop(columns=["Churn"])
try:
    pv3.fit(bad_df)
except Exception as e:
    print(f"Schema error caught: {type(e).__name__}: {e}")

---
## Step 4: Extract Service Classes

**Goal**: Each class does one thing. The pipeline orchestrates, it doesn't implement.

A well-designed pipeline looks like:

In [ ]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class PipelineConfig:
    n_estimators: int = 100
    test_size: float = 0.2
    random_state: int = 42
    feature_columns: list = field(default_factory=lambda: [
        "gender", "tenure", "MonthlyCharges", "TotalCharges"
    ])
    categorical_columns: list = field(default_factory=lambda: [
        "gender", "Partner", "Dependents", "PhoneService", "Churn"
    ])
    target_column: str = "Churn"
    max_missing_pct: float = 0.15


class DataCleaner:
    """Responsible only for cleaning: drop ID, fix TotalCharges, drop NaN."""

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(columns=["customerID"], errors="ignore").copy()
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        return df.dropna()


class CategoricalEncoder:
    """Responsible only for encoding categorical columns."""

    def __init__(self, columns: list):
        self.columns = columns

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for col in self.columns:
            if col in df.columns:
                df[col] = LabelEncoder().fit_transform(df[col].astype(str))
        return df


class ModelTrainer:
    """Responsible only for training and predicting."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self._model: Optional[RandomForestClassifier] = None

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "ModelTrainer":
        self._model = RandomForestClassifier(
            n_estimators=self.config.n_estimators,
            random_state=self.config.random_state,
        )
        self._model.fit(X, y)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if self._model is None: raise RuntimeError("fit() first.")
        return self._model.predict(X)


class ChurnPipelineV4:
    """Orchestrates DataCleaner, CategoricalEncoder, and ModelTrainer."""

    def __init__(self, config: PipelineConfig = None):
        self.config = config or PipelineConfig()
        self._cleaner  = DataCleaner()
        self._encoder  = CategoricalEncoder(self.config.categorical_columns)
        self._trainer  = ModelTrainer(self.config)
        self._X_test = None
        self._y_test = None

    def fit(self, df: pd.DataFrame) -> "ChurnPipelineV4":
        df = self._cleaner.clean(df)
        df = self._encoder.fit_transform(df)
        X = df[self.config.feature_columns]
        y = df[self.config.target_column]
        X_tr, self._X_test, y_tr, self._y_test = train_test_split(
            X, y, test_size=self.config.test_size, random_state=self.config.random_state
        )
        self._trainer.fit(X_tr, y_tr)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return self._trainer.predict(X)

    def report(self) -> None:
        if self._X_test is None: raise RuntimeError("Call fit() first.")
        preds = self.predict(self._X_test)
        print(classification_report(self._y_test, preds))


config = PipelineConfig(n_estimators=50)
pv4 = ChurnPipelineV4(config)
pv4.fit(df)
pv4.report()

---
## Step 5: Add Type Hints, Docstrings, and Logging

**Goal**: The code is now clear to a reader who wasn't there when you wrote it.

This is the version that belongs in `src/`. The exercise is to add the missing docstrings and logging calls to `ChurnPipelineV4` — run it through `help()` when you're done and see if a new colleague could use it from the docs alone.

In [ ]:
import logging
import time

log = logging.getLogger("churn_pipeline")


class ChurnPipelineV5(ChurnPipelineV4):
    """End-to-end churn prediction pipeline.

    Wraps data cleaning, encoding, and model training into a single
    fit/predict interface compatible with sklearn conventions.

    Example:
        >>> config = PipelineConfig(n_estimators=200)
        >>> pipeline = ChurnPipelineV5(config)
        >>> pipeline.fit(df_train)
        >>> pipeline.report()
    """

    def fit(self, df: pd.DataFrame) -> "ChurnPipelineV5":
        """Fit the full pipeline on a raw churn DataFrame.

        Args:
            df: Raw DataFrame. Must contain columns defined in config.

        Returns:
            self, to allow method chaining.

        Raises:
            KeyError: If required feature columns are missing after cleaning.
        """
        t0 = time.time()
        log.info(f"Pipeline fit started | {len(df)} rows")

        result = super().fit(df)

        log.info(f"Pipeline fit complete in {time.time()-t0:.2f}s")
        return result

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Generate predictions for a feature matrix.

        Args:
            X: Feature DataFrame. Columns must match config.feature_columns.

        Returns:
            1-D array of predicted labels (0 or 1).

        Raises:
            RuntimeError: If the pipeline has not been fitted.
        """
        log.debug(f"Predicting {len(X)} rows")
        return super().predict(X)


help(ChurnPipelineV5.fit)

---
## Step 7: Your Turn — What Would You Improve Next?

You've applied:
- Classes and encapsulation
- Error handling with a custom exception hierarchy
- Service class decomposition
- Configuration with dataclasses
- Docstrings, type hints, and logging

**What would you do to make this production-ready?** Some options:

- **Tests**: write `pytest` tests for each service class
- **CLI**: add a `main()` entry point so the pipeline runs from the command line
- **Data versioning**: track the input file with DVC
- **Experiment tracking**: log config + metrics to MLflow or W&B
- **Serialisation**: add `save(path)` / `load(path)` to the pipeline
- **REST API**: wrap `predict()` in a FastAPI endpoint

Pick one and implement it below.

In [ ]:
# Your production improvement here

# Example starter for CLI:
#
# def main():
#     import argparse
#     parser = argparse.ArgumentParser(description="Train the churn model")
#     parser.add_argument("--data", required=True, help="Path to raw CSV")
#     parser.add_argument("--n-estimators", type=int, default=100)
#     args = parser.parse_args()
#
#     df = pd.read_csv(args.data)
#     config = PipelineConfig(n_estimators=args.n_estimators)
#     pipeline = ChurnPipelineV5(config)
#     pipeline.fit(df)
#     pipeline.report()
#
# if __name__ == "__main__":
#     main()

print("What would YOU improve?")